# JSON Normalization Practice Guide

Learn how to work with JSON file and convert them to pandas DataFrames.

In [ ]:
import json
import pandas as pd
import os

print('Setup complete!')

## Reading JSON: The Common Mistake

In [ ]:
# WRONG: f.read() returns a STRING
# with open('file.json', 'r') as f:
#     data = f.read()  # This is a STRING, not a dict!
#     df = pd.json_normalize(data)  # ERROR!

# CORRECT: Use json.loads() to parse the string
# with open('file.json', 'r') as f:
#     raw = f.read()  # STRING
#     data = json.loads(raw)  # Convert to DICT/LIST
#     df = pd.json_normalize(data)  # Now it works!

print('See code comments for correct pattern')

## Simple Example: List of Objects

In [1]:
import json
import pandas as pd

# Sample JSON as string
json_string = '''[
  {"id": 1, "name": "Alice", "email": "alice@example.com"},
  {"id": 2, "name": "Bob", "email": "bob@example.com"}
]'''

# Parse string to list
data = json.loads(json_string)
print('Type:', type(data))

# Normalize to DataFrame
df = pd.json_normalize(data)
print(df)

Type: <class 'list'>
   id   name              email
0   1  Alice  alice@example.com
1   2    Bob    bob@example.com


## Nested JSON with record_path

In [4]:
# JSON with nested structure
nested_json = '''{
  "store": "store-1",
  "orders": [
    {"order_id": "o1", "items": [{"sku": "a", "qty": 1}, {"sku": "b", "qty": 2}]},
    {"order_id": "o2", "items": [{"sku": "c", "qty": 3}]}
  ]
}'''

data = json.loads(nested_json)

# Flatten with record_path and meta
df = pd.json_normalize(data, record_path=['orders', 'items'], meta=[['orders', 'order_id'], 'store'], errors='ignore')
print(df)

  sku  qty orders.order_id    store
0   a    1              o1  store-1
1   b    2              o1  store-1
2   c    3              o2  store-1


## Reading Your CA_category_id.json

In [9]:
import json
import pandas as pd
import os

# Path to your JSON file
file_path = './Streaming_platform_Data_Pipeline/Data1/CA_category_id.json'

if os.path.exists(file_path):
    # CORRECT WAY: parse with json.loads()
    with open(file_path, 'r', encoding='utf-8') as f:
        raw_string = f.read()  # Get string
        data = json.loads(raw_string)  # Parse to dict/list
    
    print('Data type:', type(data))
    print('Keys:', list(data.keys()) if isinstance(data, dict) else 'List')
    
    # Normalize based on structure
    if isinstance(data, dict) and 'items' in data:
        df = pd.json_normalize(data['items'])
    elif isinstance(data, list):
        df = pd.json_normalize(data)
    else:
        df = pd.json_normalize(data)
    
    print('\nDataFrame shape:', df.shape)
    print(df.head())
else:
    print(f'File not found at {file_path}')

Data type: <class 'dict'>
Keys: ['kind', 'etag', 'items']

DataFrame shape: (31, 6)
                    kind                                               etag  \
0  youtube#videoCategory  "ld9biNPKjAjgjV7EZ4EKeEGrhao/Xy1mB4_yLrHy_BmKm...   
1  youtube#videoCategory  "ld9biNPKjAjgjV7EZ4EKeEGrhao/UZ1oLIIz2dxIhO45Z...   
2  youtube#videoCategory  "ld9biNPKjAjgjV7EZ4EKeEGrhao/nqRIq97-xe5XRZTxb...   
3  youtube#videoCategory  "ld9biNPKjAjgjV7EZ4EKeEGrhao/HwXKamM1Q20q9BN-o...   
4  youtube#videoCategory  "ld9biNPKjAjgjV7EZ4EKeEGrhao/9GQMSRjrZdHeb1OEM...   

   id         snippet.channelId     snippet.title  snippet.assignable  
0   1  UCBR8-60-B28hp2BmDPdntcQ  Film & Animation                True  
1   2  UCBR8-60-B28hp2BmDPdntcQ  Autos & Vehicles                True  
2  10  UCBR8-60-B28hp2BmDPdntcQ             Music                True  
3  15  UCBR8-60-B28hp2BmDPdntcQ    Pets & Animals                True  
4  17  UCBR8-60-B28hp2BmDPdntcQ            Sports                True  


## Practice: Exploding Arrays

In [11]:
import pandas as pd
import json

json_str = '''[
  {"id": 1, "tags": ["a", "b", "c"]},
  {"id": 2, "tags": ["b", "d"]}
]'''

data = json.loads(json_str)
df = pd.json_normalize(data)
print('Before explode:')
print(df)

print('\nAfter explode (one tag per row):')
df_exploded = df.explode('tags')
print(df_exploded)

Before explode:
   id       tags
0   1  [a, b, c]
1   2     [b, d]

After explode (one tag per row):
   id tags
0   1    a
0   1    b
0   1    c
1   2    b
1   2    d


## Practice: Handling Missing Fields

In [12]:
import json
import pandas as pd

json_str = '''[
  {"id": 1, "name": "Alice", "email": "alice@ex.com"},
  {"id": 2, "name": "Bob"},
  {"id": 3, "email": "charlie@ex.com"}
]'''

data = json.loads(json_str)
df = pd.json_normalize(data)
print('Raw (with NaN for missing fields):')
print(df)

print('\nAfter fillna:')
df_filled = df.fillna({'name': 'Unknown', 'email': 'no-email@ex.com'})
print(df_filled)

Raw (with NaN for missing fields):
   id   name           email
0   1  Alice    alice@ex.com
1   2    Bob             NaN
2   3    NaN  charlie@ex.com

After fillna:
   id     name            email
0   1    Alice     alice@ex.com
1   2      Bob  no-email@ex.com
2   3  Unknown   charlie@ex.com


## Summary
1. **Key rule**: f.read() returns STRING → use json.loads() to parse
2. **Simple lists**: pd.json_normalize(list_of_dicts)
3. **Nested records**: use record_path + meta
4. **Arrays**: use df.explode(column_name)
5. **Missing fields**: NaN filled automatically, use fillna() to replace
6. **Double-encoded**: detect string starting with '{' or '[' and json.loads() it again

In [5]:
import json
import pandas as pd

json_str = ''' 
{
    "name": "John",
    "age": 30,
    "city": "New York",
    "children": [
        {
            "name": "Jane",
            "age": 10
        },
        {
            "name": "Doe",
            "age": 5
        }
    ]
}
'''

data=json.loads(json_str)
df = pd.json_normalize(data,
                       record_path = ['children'],
                       meta=['name','age' ,'city'],
                       meta_prefix='parent_'
                       )
print(df)

   name  age parent_name parent_age parent_city
0  Jane   10        John         30    New York
1   Doe    5        John         30    New York


In [8]:
json_str = ''' 
[

{

"user_id": 1,
"activity": [

{"action": "view", "timestamp": "2025-01-01T10:00:00", "product_id": 101},
{"action": "add_to_cart", "timestamp": "2025-01-01T10:05:00", "product_id": 101},
{"action": "purchase", "timestamp": "2025-01-01T10:10:00", "product_id": 101}

]

},
{

"user_id": 2,
"activity": [

{"action": "view", "timestamp": "2025-01-01T11:00:00", "product_id": 102},
{"action": "view", "timestamp": "2025-01-01T11:05:00", "product_id": 103}

]

},
{

"user_id": 3,
"activity": [

{"action": "view", "timestamp": "2025-01-01T12:00:00", "product_id": 104},
{"action": "add_to_cart", "timestamp": "2025-01-01T12:05:00", "product_id": 104}

]

}
]
'''
data = json.loads(json_str)
df = pd.json_normalize(data,
                       record_path = ['activity'],
                       meta=['user_id'])
print(df.reindex(columns=['user_id','product_id', 'action', 'timestamp']))

  user_id  product_id       action            timestamp
0       1         101         view  2025-01-01T10:00:00
1       1         101  add_to_cart  2025-01-01T10:05:00
2       1         101     purchase  2025-01-01T10:10:00
3       2         102         view  2025-01-01T11:00:00
4       2         103         view  2025-01-01T11:05:00
5       3         104         view  2025-01-01T12:00:00
6       3         104  add_to_cart  2025-01-01T12:05:00


In [31]:
# filter out user who have NOT performed    at least one purchase action
#result = df.groupby('user_id')['action'].apply(lambda x: 'purchase' not in x.values)
purchased_users = df[df.action== 'purchase']['user_id'].unique()
print('Users who made a purchase:', purchased_users)

all_users = df['user_id'].unique()
non_purchasers = [u for u in all_users if u not in purchased_users]
print('Users who did NOT make a purchase:', non_purchasers) 

Users who made a purchase: [1]
Users who did NOT make a purchase: [2, 3]


In [52]:
df1 = df.groupby(['user_id'])['action'].apply(lambda x: 'purchase' not in x.values)
print(df1)

df2 = result[df1 == True]
print(df2)

user_id
1    False
2     True
3     True
Name: action, dtype: bool
user_id
2    True
3    True
Name: action, dtype: bool
